# ORCA — Pipeline de limpieza, normalización y calidad

**Observatorio de Riesgo en Contratación de Antioquia (ORCA)**  
**Corte de datos:** 23 de octubre de 2026

Este notebook documenta y ejecuta la etapa **raw → curated** del pipeline de ORCA. La lógica de transformación permanece modularizada en `src/limpieza.py` y `src/calidad.py`; el notebook no duplica esa lógica, sino que la ejecuta, valida y deja evidencia.

### Pipeline del proyecto

```text
Fuentes oficiales
    │
    ├── DS-01 SECOP II Contratos
    ├── DS-02 SECOP II Procesos
    └── DS-03 DIVIPOLA
          │
          ▼
src/descargar_fuentes.py
          │
          ▼
       data/raw
          │
          ▼
src/limpieza.py
          │
          ├── nulos disfrazados
          ├── tipado
          ├── normalización de entidades
          ├── homologación de municipios con DIVIPOLA
          ├── medición de calidad
          ├── banderas de riesgo/calidad
          ├── seudonimización y protección de PII
          └── manifest + SHA-256
          │
          ▼
     data/curated
          │
          ▼
02_eda_inicial.ipynb
          │
          ▼
visualización / dashboard / ORCA
```

El Plan de Gobierno de Datos define `raw → curated → mart` como el linaje objetivo y exige reglas de calidad ejecutadas por lote con *quality gate* ≥ 95 %. 


## 1. Alcance y estado del pipeline

| Etapa | Implementación | Estado al 23-oct |
|---|---|---|
| Ingesta reproducible | `src/descargar_fuentes.py` | ✅ |
| Raw | `data/raw/` | ✅ |
| Limpieza y normalización DS-01 | `src/limpieza.py` | ✅ |
| Calidad | `src/calidad.py` | ✅ |
| Curated | salida de `src/limpieza.py` | 🔄 ejecutar |
| EDA inicial | `02_eda_inicial.ipynb` | ⏳ siguiente |
| Automatización | `src/monitoreo_fuente.py` + GitHub Actions | ✅ |
| Documentación / linaje | manifiestos + Plan de Gobierno | 🔄 consolidar |

### Correctivos ORCA que este notebook debe evidenciar

1. **Normalización de entidades y municipios:** cruce de `ciudad` con DIVIPOLA (DS-03), corrigiendo tildes y variantes.
2. **Depuración de fechas e incoherencias:** etiquetar inconsistencias cronológicas, incluyendo inicio anterior a firma, fin anterior a inicio y duraciones/prórrogas atípicas.
3. **Estandarización de identificación:** extraer el NIT de entidad como **9 dígitos sin dígito de verificación**, antes de cualquier seudonimización.

Estos tres puntos corresponden al plan de acción y correctivos de calidad de ORCA.


In [3]:
from pathlib import Path
import json
import glob
import pandas as pd
import numpy as np

ROOT = Path.cwd()

if not (ROOT / "notebooks").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
CURATED = ROOT / "data" / "curated"
CALIDAD = ROOT / "documentacion" / "calidad"
NOTEBOOKS = ROOT / "notebooks"

assert (NOTEBOOKS / "limpieza.py").exists(), "Falta notebooks/limpieza.py"
assert (NOTEBOOKS / "calidad.py").exists(), "Falta notebooks/calidad.py"
assert RAW.exists(), "Falta data/raw"

CURATED.mkdir(parents=True, exist_ok=True)
CALIDAD.mkdir(parents=True, exist_ok=True)

print("Raíz:", ROOT.resolve())
print("Raw:", RAW.resolve())
print("Curated:", CURATED.resolve())
print("Notebooks:", NOTEBOOKS.resolve())


Raíz: C:\Users\57317\Desktop\Data Office Strategy\Proyecto Final - Data Office Strategy\DataOfficeStrategy_Project_202620UPB
Raw: C:\Users\57317\Desktop\Data Office Strategy\Proyecto Final - Data Office Strategy\DataOfficeStrategy_Project_202620UPB\data\raw
Curated: C:\Users\57317\Desktop\Data Office Strategy\Proyecto Final - Data Office Strategy\DataOfficeStrategy_Project_202620UPB\data\curated
Notebooks: C:\Users\57317\Desktop\Data Office Strategy\Proyecto Final - Data Office Strategy\DataOfficeStrategy_Project_202620UPB\notebooks


In [4]:
# Inventario reproducible de las fuentes descargadas
raw_files = sorted(RAW.glob("DS-*.parquet"))

inventario = []
for f in raw_files:
    df_tmp = pd.read_parquet(f)
    inventario.append({
        "archivo": f.name,
        "filas": len(df_tmp),
        "columnas": len(df_tmp.columns),
        "tamaño_mb": round(f.stat().st_size / (1024**2), 2)
    })

inventario_df = pd.DataFrame(inventario)
display(inventario_df)


,archivo,filas,columnas,tamaño_mb
0,DS-01_secop2_contratos_20261023.parquet,291719,95,106.51
1,DS-02_secop2_procesos_20261023.parquet,590648,57,129.74
2,DS-03_divipola_municipios_20261023.parquet,1122,7,0.04


## 2. Ejecución de `raw → curated`

La implementación existente realiza, en orden:

1. lectura del lote raw;
2. normalización de nulos disfrazados;
3. tipado de fechas, numéricos y booleanos;
4. normalización de entidades;
5. homologación territorial con DIVIPOLA;
6. medición de calidad antes de las correcciones;
7. deduplicación por `id_contrato`;
8. creación de banderas analíticas;
9. seudonimización HMAC;
10. eliminación de PII/campos sin finalidad;
11. enmascaramiento de documentos en texto libre;
12. verificación de PII en claro;
13. escritura de Parquet, reporte de calidad y manifiesto.

La medición **antes de corregir** es intencional: permite distinguir la calidad del lote recibido de la calidad de la salida transformada.


In [5]:
import sys
NOTEBOOKS = ROOT / "notebooks"
if str(NOTEBOOKS) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS))

from limpieza import procesar

resultado = procesar(
    contratos=str(RAW / "DS-01_*.parquet"),
    divipola=str(RAW / "DS-03_*.parquet"),
    salida=str(CURATED),
    reporte=str(CALIDAD),
    corte="2026-10-23"
)

print(json.dumps(resultado, ensure_ascii=False, indent=2))


{
  "capa": "curated",
  "origen": "c:\\Users\\57317\\Desktop\\Data Office Strategy\\Proyecto Final - Data Office Strategy\\DataOfficeStrategy_Project_202620UPB\\data\\raw\\DS-01_*.parquet",
  "origen_filas": 291719,
  "filas_curated": 291719,
  "columnas_origen": 95,
  "columnas_curated": 76,
  "inicio_utc": "2026-10-23T17:10:15+00:00",
  "fin_utc": "2026-10-23T17:10:56+00:00",
  "corte": "2026-10-23",
  "calidad": {
    "puntaje_por_dimension": {
      "Completitud": 98.61,
      "Validez": 99.92,
      "Exactitud": 99.51,
      "Consistencia": 99.94,
      "Unicidad": 100.0,
      "Oportunidad": 100.0
    },
    "puntaje_lote": 99.66,
    "umbral": 95.0,
    "decision": "PUBLICAR"
  },
  "duplicados_eliminados": 0,
  "nombres_de_entidad_unificados": 32758,
  "columnas_eliminadas_pii_o_sin_finalidad": [
    "documento_proveedor",
    "identificaci_n_representante_legal",
    "n_mero_de_documento_supervisor",
    "n_mero_de_documento_ordenador_del_gasto",
    "n_mero_de_documento_orde

## 3. Resultado de la transformación

Este resumen permite evidenciar cuánto cambió el lote y qué controles de privacidad se aplicaron.


In [6]:
claves = [
    "origen_filas", "filas_curated", "columnas_origen", "columnas_curated",
    "duplicados_eliminados", "nombres_de_entidad_unificados",
    "celdas_de_texto_libre_enmascaradas", "pii_en_claro_en_curated"
]
resumen_transformacion = pd.Series(
    {k: resultado.get(k) for k in claves},
    name="valor"
).to_frame()

display(resumen_transformacion)


,valor
origen_filas,291719
filas_curated,291719
columnas_origen,95
columnas_curated,76
duplicados_eliminados,0
nombres_de_entidad_unificados,32758
celdas_de_texto_libre_enmascaradas,24
pii_en_claro_en_curated,34


## 4. Quality Gate de ORCA

ORCA define seis dimensiones de calidad:

- Completitud
- Validez
- Exactitud
- Consistencia
- Unicidad
- Oportunidad

El puntaje del lote es el promedio de las seis dimensiones.

**Umbral de publicación: 95 %.**

- `PUBLICAR` si puntaje ≥ 95 %
- `CUARENTENA` si puntaje < 95 %

Las reglas se encuentran en `src/calidad.py` y son las que deben ser aprobadas por el Data Steward según el Plan de Gobierno.


In [7]:
calidad_resultado = resultado["calidad"]

tabla_calidad = pd.DataFrame([
    {
        "dimension": d,
        "cumplimiento_pct": v
    }
    for d, v in calidad_resultado["puntaje_por_dimension"].items()
])

display(tabla_calidad)

print(f"Puntaje del lote: {calidad_resultado['puntaje_lote']}%")
print(f"Umbral: {calidad_resultado['umbral']}%")
print(f"Decisión: {calidad_resultado['decision']}")


,dimension,cumplimiento_pct
0,Completitud,98.61
1,Validez,99.92
2,Exactitud,99.51
3,Consistencia,99.94
4,Unicidad,100.00
5,Oportunidad,100.00


Puntaje del lote: 99.66%
Umbral: 95.0%
Decisión: PUBLICAR


In [8]:
# Evidencia detallada de cada regla
reporte_path = CALIDAD / "reporte_calidad_DS-01.csv"
assert reporte_path.exists(), f"No existe {reporte_path}"

reporte_calidad = pd.read_csv(reporte_path)
display(reporte_calidad)


,dimension,regla,descripcion,evaluados,cumplen,fallan,cumplimiento_pct
0,Completitud,C1,fecha_de_firma presente,291719,291719,0,100.00
1,Completitud,C2,valor_del_contrato presente,291719,291719,0,100.00
2,Completitud,C3,documento_proveedor presente,291719,291165,554,99.81
3,Completitud,C4,ciudad definida (no 'No Definido'),291719,272024,19695,93.25
4,Completitud,C5,modalidad_de_contratacion presente,291719,291657,62,99.98
5,Validez,V1,valor_del_contrato no negativo,291719,291719,0,100.00
6,Validez,V2,fecha_de_firma no es futura,291719,291719,0,100.00
7,Validez,V3,ciudad reconocida en DIVIPOLA,272024,271395,629,99.77
8,Exactitud,E1,valor_del_contrato mayor que cero,291719,288910,2809,99.04
9,Exactitud,E2,valor_del_contrato sin atípico (<= $100.000 mi...,291719,291672,47,99.98


## 5. Correctivo 1 — Normalización de entidades y municipios

El Plan de Gobierno identifica como problema los valores `No Definido` y establece el cruce con **DIVIPOLA (DS-03)** como correctivo.

La implementación de `src/limpieza.py`:

- elimina tildes y normaliza mayúsculas;
- aplica alias conocidos;
- cruza la ciudad normalizada contra DIVIPOLA;
- genera `cod_municipio` y `municipio`;
- evita asignar automáticamente variantes ambiguas.

La métrica que importa para ORCA es la cobertura del cruce, no simplemente el cambio de formato textual.


In [9]:
# Validación independiente de la homologación territorial sobre curated
df_curated = pd.read_parquet(CURATED / "DS-01_contratos_curated.parquet")

cols_municipio = [c for c in ["ciudad", "cod_municipio", "municipio"] if c in df_curated.columns]
display(df_curated[cols_municipio].head(10))

municipio_eval = pd.DataFrame({
    "indicador": [
        "Filas curated",
        "Ciudades con cod_municipio",
        "Ciudades sin cod_municipio",
        "Cobertura DIVIPOLA (%)"
    ],
    "valor": [
        len(df_curated),
        int(df_curated["cod_municipio"].notna().sum()),
        int(df_curated["cod_municipio"].isna().sum()),
        round(100 * df_curated["cod_municipio"].notna().mean(), 2)
    ]
})
display(municipio_eval)


,ciudad,cod_municipio,municipio
0,Medellín,05001,MEDELLÍN
1,Medellín,05001,MEDELLÍN
2,Hispania,05353,HISPANIA
3,San Pedro De Uraba,05665,SAN PEDRO DE URABÁ
4,Bello,05088,BELLO
5,Medellín,05001,MEDELLÍN
6,Apartadó,05045,APARTADÓ
7,Copacabana,05212,COPACABANA
8,Medellín,05001,MEDELLÍN
9,Rionegro,05615,RIONEGRO


,indicador,valor
0,Filas curated,291719.00
1,Ciudades con cod_municipio,271395.00
2,Ciudades sin cod_municipio,20324.00
3,Cobertura DIVIPOLA (%),93.03


## 6. Correctivo 2 — Depuración de fechas e incoherencias

ORCA no debe borrar silenciosamente los registros problemáticos. Debe **etiquetarlos** para que el analista pueda priorizarlos.

Se revisan, como mínimo:

- `fecha_de_inicio_del_contrato < fecha_de_firma`;
- `fecha_de_fin_del_contrato < fecha_de_inicio_del_contrato`;
- `dias_adicionados < 0`;
- `dias_adicionados > 3.650`;
- valores extremos que puedan corresponder a errores de digitación.

Esto permite conservar el dato de origen y convertir la anomalía en una bandera analítica.


In [10]:
# Revisión de inconsistencias cronológicas y prórrogas atípicas
# Se usa la salida curated; si una columna no está disponible, se reporta explícitamente.

df_eval = df_curated.copy()

fechas = [
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato"
]

for c in fechas:
    if c in df_eval.columns:
        df_eval[c] = pd.to_datetime(df_eval[c], errors="coerce")

if all(c in df_eval.columns for c in fechas):
    df_eval["flag_inicio_antes_firma"] = (
        df_eval["fecha_de_inicio_del_contrato"] < df_eval["fecha_de_firma"]
    ).fillna(False)

    df_eval["flag_fin_antes_inicio"] = (
        df_eval["fecha_de_fin_del_contrato"] < df_eval["fecha_de_inicio_del_contrato"]
    ).fillna(False)

if "dias_adicionados" in df_eval.columns:
    df_eval["dias_adicionados"] = pd.to_numeric(
        df_eval["dias_adicionados"], errors="coerce"
    )
    df_eval["flag_prorroga_negativa"] = (df_eval["dias_adicionados"] < 0).fillna(False)
    df_eval["flag_prorroga_atipica"] = (df_eval["dias_adicionados"] > 3650).fillna(False)

flags = {
    c: int(df_eval[c].sum())
    for c in [
        "flag_inicio_antes_firma",
        "flag_fin_antes_inicio",
        "flag_prorroga_negativa",
        "flag_prorroga_atipica"
    ]
    if c in df_eval.columns
}

pd.DataFrame(
    [{"indicador": k, "registros": v} for k, v in flags.items()]
)


,indicador,registros
0,flag_inicio_antes_firma,11
1,flag_fin_antes_inicio,85
2,flag_prorroga_negativa,0
3,flag_prorroga_atipica,7


### Nota de implementación

Esta validación es deliberadamente más amplia que una simple eliminación de filas. Si alguna de las banderas anteriores todavía no se materializa dentro de `src/limpieza.py`, el resultado de esta celda debe utilizarse como evidencia para completar la implementación antes de considerar cerrado el correctivo.

**No se debe ocultar una inconsistencia para hacer subir artificialmente el quality gate.**


## 7. Correctivo 3 — Estandarización del NIT

El Plan de Gobierno establece que el identificador de entidad debe quedar en **9 dígitos, eliminando el dígito de verificación**, antes de cualquier seudonimización.

La estandarización debe:

1. conservar únicamente dígitos;
2. validar que existan al menos 10 dígitos cuando la fuente incluye el dígito de verificación;
3. retirar el último dígito;
4. conservar el resultado como texto para no perder ceros a la izquierda;
5. no aplicar esta regla a identificadores de personas naturales.


In [15]:
def nit_9_digitos(serie: pd.Series) -> pd.Series:
    """Normaliza el NIT a 9 dígitos base."""
    s = serie.astype("string").str.replace(r"\D", "", regex=True)

    # Si ya tiene 9 dígitos, se conserva.
    # Si tiene 10 o más, se retira el último dígito (DV).
    return s.where(
        s.str.len() == 9,
        s.where(s.str.len() >= 10).str[:-1]
    )


# Validación sobre el raw original
raw_ds01 = pd.read_parquet(sorted(RAW.glob("DS-01_*.parquet"))[-1])

raw_ds01["id_nit_entidad_estandarizado"] = nit_9_digitos(
    raw_ds01["nit_entidad"]
)

nit_digitos = (
    raw_ds01["nit_entidad"]
    .astype("string")
    .str.replace(r"\D", "", regex=True)
    .str.len()
)

nit_eval = pd.DataFrame({
    "indicador": [
        "NIT con valor en raw",
        "NIT ya estandarizado a 9 dígitos",
        "NIT con 10 dígitos (9 + DV)",
        "NIT con formato distinto de 9 o 10 dígitos",
        "NIT finalmente estandarizados a 9 dígitos"
    ],
    "registros": [
        int(raw_ds01["nit_entidad"].notna().sum()),
        int((nit_digitos == 9).sum()),
        int((nit_digitos == 10).sum()),
        int((~nit_digitos.isin([9, 10])).sum()),
        int(raw_ds01["id_nit_entidad_estandarizado"].notna().sum())
    ]
})

display(nit_eval)

display(
    raw_ds01[
        ["nit_entidad", "id_nit_entidad_estandarizado"]
    ]
    .dropna()
    .head(10)
)


,indicador,registros
0,NIT con valor en raw,291719
1,NIT ya estandarizado a 9 dígitos,267504
2,NIT con 10 dígitos (9 + DV),24215
3,NIT con formato distinto de 9 o 10 dígitos,0
4,NIT finalmente estandarizados a 9 dígitos,291719


,nit_entidad,id_nit_entidad_estandarizado
0,899999239,899999239
1,800194096,800194096
2,890984986,890984986
3,890983814,890983814
4,890980112,890980112
5,899999239,899999239
6,8909800952,890980095
7,8909807673,890980767
8,800194096,800194096
9,899999034,899999034


## 8. Validación de la capa curated y privacidad

La salida curated debe:

- ser legible;
- conservar las variables estratégicas necesarias;
- tener `id_contrato` único;
- no exponer PII en claro;
- contar con manifiesto y SHA-256;
- quedar separada de `data/raw`.

La capa raw contiene PII en claro y no se versiona en GitHub. La capa curated es la que alimentará el EDA.


In [12]:
curated_path = CURATED / "DS-01_contratos_curated.parquet"
manifest_path = CURATED / "DS-01_contratos_curated.manifest.json"

assert curated_path.exists(), f"No existe {curated_path}"
assert manifest_path.exists(), f"No existe {manifest_path}"

with open(manifest_path, encoding="utf-8") as f:
    manifest = json.load(f)

columnas_clave = [
    "id_contrato",
    "nit_entidad",
    "fecha_de_firma",
    "valor_del_contrato",
    "modalidad_de_contratacion"
]
faltantes = [c for c in columnas_clave if c not in df_curated.columns]

validacion = pd.DataFrame({
    "control": [
        "Filas curated",
        "Columnas curated",
        "Duplicados id_contrato",
        "PII en claro reportada por pipeline",
        "Columnas estratégicas faltantes",
        "SHA-256 disponible"
    ],
    "resultado": [
        len(df_curated),
        len(df_curated.columns),
        int(df_curated["id_contrato"].duplicated().sum()),
        resultado["pii_en_claro_en_curated"],
        len(faltantes),
        bool(manifest.get("sha256_curated"))
    ]
})

display(validacion)


,control,resultado
0,Filas curated,291719
1,Columnas curated,76
2,Duplicados id_contrato,0
3,PII en claro reportada por pipeline,34
4,Columnas estratégicas faltantes,0
5,SHA-256 disponible,True


In [13]:
# Lectura del manifiesto de linaje
manifest_view = {
    k: manifest.get(k)
    for k in [
        "capa", "origen", "origen_filas", "filas_curated",
        "columnas_origen", "columnas_curated", "corte",
        "duplicados_eliminados", "nombres_de_entidad_unificados",
        "celdas_de_texto_libre_enmascaradas",
        "pii_en_claro_en_curated", "sha256_curated",
        "clasificacion"
    ]
}
display(pd.Series(manifest_view, name="valor").to_frame())


,valor
capa,curated
origen,c:\Users\57317\Desktop\Data Office Strategy\Pr...
origen_filas,291719
filas_curated,291719
columnas_origen,95
columnas_curated,76
corte,2026-10-23
duplicados_eliminados,0
nombres_de_entidad_unificados,32758
celdas_de_texto_libre_enmascaradas,24


## 9. Cierre de la etapa y criterios para pasar al EDA

La etapa **raw → curated** se considera lista para EDA cuando:

- [ ] se generó `DS-01_contratos_curated.parquet`;
- [ ] el quality gate fue calculado con las seis dimensiones;
- [ ] la decisión del lote quedó registrada;
- [ ] la homologación DIVIPOLA quedó medida;
- [ ] las inconsistencias de fechas quedaron etiquetadas;
- [ ] la estandarización del NIT quedó implementada en el pipeline, no solo calculada como prueba;
- [ ] `PII en claro = 0` en curated;
- [ ] existe manifiesto con SHA-256;
- [ ] los raw no se suben al repositorio.

### Siguiente archivo

**`02_eda_inicial.ipynb`** trabajará sobre `data/curated/` y combinará, cuando corresponda, DS-02 para responder las preguntas de negocio de ORCA.

No se debe volver a limpiar el dato dentro del EDA.


## Conclusión

La arquitectura final queda separada por responsabilidad:

**ingesta → raw → limpieza/calidad → curated → EDA → visualización/dashboard**

La lógica de producción permanece en `src/`. Los notebooks sirven para **orquestar, validar, analizar y dejar evidencia**.

Esto permite que ORCA sea reproducible: si mañana cambia la fuente, se genera un nuevo lote raw, se vuelve a ejecutar el pipeline y se obtiene un nuevo resultado de calidad y un nuevo manifiesto de linaje.
